# SQL basics

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-01-basics.ipynb)

SQL (Structured Query Language) is how you ask a database questions. You describe the rows you want, and the
engine works out how to find them. This notebook covers the everyday building blocks: choosing columns,
filtering rows, sorting, summarising, grouping, combining tables, conditional values and missing values.

The examples run in Spark SQL, the SQL dialect of Apache Spark. Most of what you learn here works unchanged in
PostgreSQL, MySQL, BigQuery, Snowflake and Redshift. The notebook ends with practice questions and their answers.

## Setup

Install PySpark and start a Spark session. In Colab this takes about a minute.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## The example data

Two small tables, `customers` and `orders`, are created below as temporary views. A view is a name for a table
that lives only for this session, so you can query it like any table.

`customers` has one row per customer. `orders` has one row per order, and `customer_id` links each order to its
customer. Customers 7 and 8 have never ordered, which the later examples rely on.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. Tables, rows and columns

A **table** is a set of rows that share the same columns. Each **row** describes one thing, here one customer or
one order. Each **column** holds one attribute of that thing, such as `name` or `amount`.

The simplest query reads every row and every column:

```sql
SELECT * FROM customers;
```

`SELECT` names the columns you want, `FROM` names the table, and `*` means "all columns". The rest of this
notebook replaces the `*` with what you actually need.

In [ ]:
spark.sql("SELECT * FROM customers").show()

## 2. Choosing columns

List the columns you want, separated by commas. Give a column a shorter or clearer name with `AS`.
`DISTINCT` removes duplicate rows from the result.

```sql
SELECT column1, column2 AS alias FROM table;
SELECT DISTINCT column FROM table;
```

In [ ]:
spark.sql("""
SELECT name, region
FROM customers
""").show()

spark.sql("""
SELECT name AS customer_name
FROM customers
""").show()

spark.sql("""
SELECT DISTINCT region
FROM customers
""").show()

## 3. Filtering rows with WHERE

`WHERE` keeps only the rows whose condition is true. It runs before anything else, so it decides which rows the
rest of the query sees.

| Operator | Meaning |
| --- | --- |
| `=`, `<>` | equal, not equal |
| `<`, `<=`, `>`, `>=` | comparisons |
| `AND`, `OR`, `NOT` | combine conditions |
| `IN (...)` | equal to any value in the list |
| `BETWEEN a AND b` | from a to b, both ends included |
| `LIKE` | text pattern: `%` is any text, `_` is one character |

`AND` binds tighter than `OR`, so use parentheses whenever you mix them.

In [ ]:
spark.sql("""
SELECT order_id, amount
FROM orders
WHERE status = 'completed' AND amount >= 300
""").show()

spark.sql("""
SELECT name, region
FROM customers
WHERE region IN ('North', 'West')
""").show()

spark.sql("""
SELECT order_id, amount
FROM orders
WHERE amount BETWEEN 150 AND 300
""").show()

spark.sql("""
SELECT name
FROM customers
WHERE name LIKE '%Corp' OR name LIKE 'Echo%'
""").show()

## 4. Sorting and limiting

`ORDER BY` sorts the result. `ASC` is ascending (the default) and `DESC` is descending. You can sort by several
columns; later columns break ties in earlier ones. `LIMIT n` returns only the first `n` rows, which is useful
together with a sort.

```sql
SELECT ... FROM table ORDER BY column1 DESC, column2 ASC LIMIT 10;
```

In Spark, `NULL` values sort first in ascending order. Add `NULLS LAST` to move them to the end.

In [ ]:
spark.sql("""
SELECT order_id, amount
FROM orders
ORDER BY amount DESC
LIMIT 3
""").show()

## 5. Summarising with aggregate functions

An **aggregate function** turns many rows into one value.

| Function | Returns |
| --- | --- |
| `COUNT(*)` | number of rows |
| `COUNT(column)` | number of rows where `column` is not `NULL` |
| `SUM(column)` | total |
| `AVG(column)` | average |
| `MIN(column)`, `MAX(column)` | smallest and largest value |

Aggregates ignore `NULL` values. Without a `GROUP BY` (next section), the whole table is one group and the query
returns one row.

In [ ]:
spark.sql("""
SELECT
  COUNT(*)                  AS orders,
  SUM(amount)               AS total_amount,
  ROUND(AVG(amount), 2)     AS average_amount,
  MIN(amount)               AS smallest,
  MAX(amount)               AS largest
FROM orders
WHERE status = 'completed'
""").show()

## 6. Grouping with GROUP BY

`GROUP BY` splits the rows into groups and computes the aggregates once per group. The result has one row per
group.

```sql
SELECT column, COUNT(*) FROM table GROUP BY column;
```

Every column in the `SELECT` list must either appear in `GROUP BY` or be wrapped in an aggregate. Selecting
`order_id` next to `GROUP BY status` is an error, because a status has many orders and the engine cannot pick one.

In [ ]:
spark.sql("""
SELECT status, COUNT(*) AS orders, SUM(amount) AS total_amount
FROM orders
GROUP BY status
ORDER BY status
""").show()

spark.sql("""
SELECT customer_id, status, COUNT(*) AS orders
FROM orders
GROUP BY customer_id, status
ORDER BY customer_id, status
""").show()

## 7. WHERE vs HAVING

`WHERE` and `HAVING` both filter, but they run at different stages. `WHERE` filters **rows** before anything is
grouped. `HAVING` filters **groups** after the aggregates are computed.

| Question | Keyword |
| --- | --- |
| Is this one order completed? | `WHERE status = 'completed'` |
| Does this customer's total exceed 500? | `HAVING SUM(amount) > 500` |
| Does this customer have more than one completed order? | `WHERE` for the status, `HAVING COUNT(*) > 1` |

A practical test: if the condition is about a single order, it goes in `WHERE`. If it is about a customer's total
or count, it goes in `HAVING`.

In [ ]:
spark.sql("""
SELECT customer_id, SUM(amount) AS completed_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
""").show()

The `status` condition is about each order, so it stays in `WHERE`. The threshold is about each customer's total, so
it goes in `HAVING`. Filtering rows first also changes the answer: rows removed by `WHERE` never reach the
aggregates.

**Common mistakes.** `WHERE SUM(amount) > 500` is rejected, because the sum does not exist yet at the `WHERE` stage.
`HAVING status = 'completed'` fails, because `status` is neither grouped nor aggregated. Move each condition to the
stage it belongs to.

## 8. How a query runs

You write a query in one order, but the engine evaluates its clauses in another:

1. `FROM` and `JOIN`: choose the tables and combine them.
2. `WHERE`: keep the rows that match.
3. `GROUP BY`: form the groups.
4. `HAVING`: keep the groups that match.
5. `SELECT`: compute the output columns.
6. `ORDER BY`: sort the result.
7. `LIMIT`: keep the first rows.

This is why an alias defined in `SELECT` cannot be used in `WHERE`: the `WHERE` step runs first. It is also why a
window or aggregate cannot appear in `WHERE`.

## 9. Combining tables with JOIN

A **join** matches rows from two tables by a key. The most common join is `INNER JOIN`, which keeps only the
matches. `LEFT JOIN` keeps every row of the left table and fills the missing right-hand columns with `NULL`.

```sql
SELECT ... FROM left_table l JOIN right_table r ON l.key = r.key;
```

In [ ]:
spark.sql("""
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
LIMIT 4
""").show()

spark.sql("""
SELECT c.name, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
""").show()

The [Joins](sql-06-joins.ipynb) notebook covers join types, join conditions and row multiplication in depth.

## 10. Conditional values with CASE

`CASE` is SQL's if/else. It checks conditions from top to bottom and returns the value of the first one that is
true. Without an `ELSE`, the result is `NULL` when nothing matches.

```sql
CASE WHEN condition THEN value WHEN condition2 THEN value2 ELSE other END
```

In [ ]:
spark.sql("""
SELECT order_id, amount,
       CASE WHEN amount >= 300 THEN 'big'
            ELSE 'small'
       END AS size
FROM orders
ORDER BY order_id
LIMIT 5
""").show()

## 11. Missing values with NULL

`NULL` means the value is unknown. It is not zero and it is not an empty string. Any comparison with `NULL` is
unknown, so `WHERE x = NULL` never matches anything. Test for missing values with `IS NULL` and `IS NOT NULL`.
`COALESCE(a, b)` returns the first value that is not `NULL`, which is how you supply a default.

In [ ]:
spark.sql("""
WITH scores AS (
  SELECT * FROM VALUES (1, 10), (2, NULL), (3, 30) AS scores(id, score)
)
SELECT id,
       score,
       score IS NULL            AS is_missing,
       COALESCE(score, 0)       AS score_or_zero
FROM scores
""").show()

## Interview notes

- **COUNT(*) vs COUNT(column).** `COUNT(*)` counts rows; `COUNT(column)` counts the non-NULL values in that column.
  The two give different answers whenever the column has NULLs.
- **WHERE vs HAVING.** Put a row condition in `WHERE`, and a condition on an aggregate in `HAVING`. Interviewers
  often ask which runs first: `WHERE`.
- **NULL in comparisons.** `x = NULL` is never true. Use `IS NULL`. The NULL chapter covers `NOT IN` with NULLs.
- **Joins.** An inner join keeps matches; a left join keeps every left row. The joins chapter covers the rest.

## Practice questions

Try each question before you open the answers. Each one uses only what this notebook covers.

1. List the names of the customers in the North region, sorted by name.
2. How many orders did each customer place? Show `customer_id` and the count, sorted by `customer_id`.
3. What is the total amount of all completed orders?
4. Which customers have more than one completed order? Show `customer_id` and the number of completed orders.
5. For each status, show the number of orders and the average amount, rounded to 2 decimals.
6. Show every order with the customer's name. Return `order_id`, `name` and `amount`, sorted by `order_id`.
7. Which customers have never placed an order? Show their names, sorted by name.
8. Label each order `big` if its amount is 300 or more and `small` otherwise. How many orders have each label?
9. What is the largest completed order? Show its `order_id` and `amount`.
10. Which customer has the highest total completed amount? Show `customer_id` and the total.

## Answers

Each answer shows the query, its result, and an `assert` that checks the result.

### Question 1

In [ ]:
q = """
SELECT name FROM customers WHERE region = 'North' ORDER BY name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Acme Corp",), ("Beta LLC",)]

### Question 2

In [ ]:
q = """
SELECT customer_id, COUNT(*) AS orders
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 3), (3, 2), (4, 2), (5, 2), (6, 2)]

Customers 7 and 8 do not appear: a `GROUP BY` on `orders` only sees customers who have orders. Question 7 handles them.

### Question 3

In [ ]:
q = """
SELECT SUM(amount) AS completed_total FROM orders WHERE status = 'completed'
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2995,)]

### Question 4

In [ ]:
q = """
SELECT customer_id, COUNT(*) AS completed_orders
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 2), (3, 2), (5, 2)]

### Question 5

In [ ]:
q = """
SELECT status, COUNT(*) AS orders, ROUND(AVG(amount), 2) AS average_amount
FROM orders
GROUP BY status
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2, 287.5), ("completed", 11, 272.27), ("pending", 1, 220.0)]

### Question 6

In [ ]:
q = """
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(5)
rows = result.collect()
assert len(rows) == 14 and tuple(rows[0]) == (1, "Acme Corp", 100)

### Question 7

In [ ]:
q = """
SELECT c.name
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Gamma SA",), ("Hotel BV",)]

### Question 8

In [ ]:
q = """
SELECT CASE WHEN amount >= 300 THEN 'big' ELSE 'small' END AS size, COUNT(*) AS orders
FROM orders
GROUP BY CASE WHEN amount >= 300 THEN 'big' ELSE 'small' END
ORDER BY size
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("big", 6), ("small", 8)]

### Question 9

In [ ]:
q = """
SELECT order_id, amount FROM orders WHERE status = 'completed' ORDER BY amount DESC LIMIT 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 500)]

### Question 10

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS completed_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY completed_total DESC
LIMIT 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(5, 950)]